In [1]:
import geopandas as gpd
import rasterio
import numpy as np
from rasterio.sample import sample_gen


In [2]:

gdf = gpd.read_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/night_cut_4326.parquet")

In [3]:
print("CRS:", gdf.crs)


CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Horiz

In [4]:
# === 1. Шлях до geoid-моделі ( EGG2015.tif) ===
geoid_path = "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_raw/raw_data/egg_2015.tif"

In [5]:
with rasterio.open(geoid_path) as src:
    dem_crs = src.crs
    print("DEM CRS:", dem_crs)

DEM CRS: EPSG:4326


In [6]:
# координати точок
coords = [(geom.x, geom.y) for geom in gdf.geometry]


In [7]:
gdf.head()

,region,sc_orient,track,segment_dist,solar_elevation,segment_id,background_rate,cycle,pair,rgt,...,y_atc,landcover,x_atc,relief,quality_ph,atl03_cnf,atl08_class,geometry,spot,time
2018-11-04 01:05:32.246215936,6.0,1.0,1.0,1.473420e+07,-40.593658,735621.0,1840.163382,1.0,0.0,556.0,...,3904.601562,255.0,-14.784105,0.0,0.0,4.0,1.0,POINT (24.75273 47.93399),6.0,2018-11-04 01:05:32.246215936
2018-11-04 01:05:32.246416128,6.0,1.0,1.0,1.473420e+07,-40.593658,735621.0,1840.163382,1.0,0.0,556.0,...,3904.586670,255.0,-13.374234,0.0,0.0,4.0,1.0,POINT (24.75273 47.93398),6.0,2018-11-04 01:05:32.246416128
2018-11-04 01:05:32.247016192,6.0,1.0,1.0,1.473420e+07,-40.593658,735621.0,1840.163382,1.0,0.0,556.0,...,3904.569580,255.0,-9.145994,0.0,0.0,4.0,2.0,POINT (24.75272 47.93394),6.0,2018-11-04 01:05:32.247016192
2018-11-04 01:05:32.247316224,6.0,1.0,1.0,1.473420e+07,-40.593658,735621.0,1840.163382,1.0,0.0,556.0,...,3904.604248,255.0,-7.035925,0.0,0.0,4.0,1.0,POINT (24.75272 47.93392),6.0,2018-11-04 01:05:32.247316224
2018-11-04 01:05:32.247916288,6.0,1.0,1.0,1.473420e+07,-40.593658,735621.0,1840.163382,1.0,0.0,556.0,...,3904.604736,255.0,-2.809901,0.0,0.0,4.0,1.0,POINT (24.75271 47.93388),6.0,2018-11-04 01:05:32.247916288


In [8]:

# витяг значень з моделі квазігеоїда
with rasterio.open(geoid_path) as geoid_src:
    geoid_values = list(sample_gen(geoid_src, coords))

# додавання геоїдної та ортометричної висоти
gdf["geoid_height"] = [val[0] if val else np.nan for val in geoid_values]
gdf["orthometric_height"] = gdf["height"] - gdf["geoid_height"]




In [9]:
# === 5. Зберегти результат (наприклад, як GeoParquet або GeoPackage) ===
gdf.to_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_night_cut_4326_orthometric.parquet")

In [12]:
gdf_utm = gdf.to_crs(epsg=32635)

In [13]:
gdf_utm.to_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_night_cut_32635_orthometric.parquet")


In [10]:
gdf_m = gdf.to_crs(epsg=3857)

In [11]:
gdf_m.to_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_night_cut_3857_orthometric.parquet")
